# Lab 17: Pix2Pix — Edges to Digits

**Goal:** Build a complete Pix2Pix system on MNIST (Unit 4 · L16–L17).

**Notation:** x = input image (the edge map), y = **target image** (the original digit) — not a class label as in L10. Pix2Pix is L10's conditional GAN, conditioned on an image.

**Task:** Given edge-detected digit images, generate the original digits.

What we'll build:
1. Create paired data: edges ↔ original digits
2. U-Net Generator (with skip connections)
3. PatchGAN Discriminator
4. Combined loss: GAN + L1
5. Train and compare: L1 only vs L1 + GAN
6. Test on unseen edges

**Runtime (3 models × 20 epochs, measured on an Apple-silicon laptop):** `mps` ≈ 0.3 min/epoch → about 15–20 min in total. Laptop CPU ≈ 4.6 min/epoch on all 60,000 images, so on CPU the notebook automatically uses `TRAIN_SUBSET = 10000` (≈ 0.8 min/epoch → about 40 min). A CUDA GPU is faster still.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(42)
print(f"Using: {device}")

## 1. Create Paired Data: Edges ↔ Digits

We need paired data for Pix2Pix. We'll create it from MNIST:
- **Input:** edge-detected version of each digit
- **Target:** the original digit

This simulates a real Pix2Pix task like "sketch → photo".

In [ ]:
def make_edges(img):
    """Simple edge detection using Sobel filters.
    Input: (1, H, W) tensor in [-1, 1]
    Output: (1, H, W) edge map in [-1, 1]"""
    # Sobel kernels
    sobel_x = torch.tensor([[-1, 0, 1],
                            [-2, 0, 2],
                            [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1,-2,-1],
                            [ 0, 0, 0],
                            [ 1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)

    img_4d = img.unsqueeze(0)  # (1, 1, H, W)
    gx = F.conv2d(img_4d, sobel_x, padding=1)
    gy = F.conv2d(img_4d, sobel_y, padding=1)
    edges = torch.sqrt(gx**2 + gy**2 + 1e-8)

    # Normalize to [-1, 1]
    edges = edges.squeeze(0)
    edges = edges / (edges.max() + 1e-8)    # [0, 1]
    edges = edges * 2 - 1                     # [-1, 1]
    return edges


class EdgeDigitDataset(Dataset):
    """Paired dataset: edge maps -> original digits."""
    def __init__(self, mnist_dataset):
        self.mnist = mnist_dataset

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        target, label = self.mnist[idx]  # target = original digit
        edges = make_edges(target)        # input = edge map
        return edges, target, label


# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
mnist_train = datasets.MNIST("data", train=True, download=True, transform=transform)
mnist_test  = datasets.MNIST("data", train=False, download=True, transform=transform)

# On a laptop CPU, train on a subset (None = all 60,000)
TRAIN_SUBSET = None if device.type != "cpu" else 10000
if TRAIN_SUBSET:
    mnist_train = torch.utils.data.Subset(mnist_train, range(TRAIN_SUBSET))

train_data = EdgeDigitDataset(mnist_train)
test_data  = EdgeDigitDataset(mnist_test)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, drop_last=True)
test_loader  = DataLoader(test_data, batch_size=64, shuffle=False)

print(f"Training pairs: {len(train_data)}")
print(f"Test pairs: {len(test_data)}")

In [ ]:
# Visualize some pairs
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    edges, target, label = train_data[i]
    axes[0, i].imshow((edges.squeeze() + 1)/2, cmap="gray")
    axes[0, i].axis("off")
    axes[0, i].set_title(str(label))
    axes[1, i].imshow((target.squeeze() + 1)/2, cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Edges\n(input)", fontsize=11, rotation=0, labelpad=40)
axes[1, 0].set_ylabel("Original\n(target)", fontsize=11, rotation=0, labelpad=40)
plt.suptitle("Paired Data: Edge Maps ↔ Original Digits", fontsize=13)
plt.tight_layout()
plt.show()

## 2. U-Net Generator

Encoder-decoder with skip connections.
Sized for 28x28 MNIST (smaller than the 256x256 in the paper).

```
Encoder:  28→14→7→3→1   (4 downsampling steps)
Decoder:  1→3→7→14→28   (4 upsampling steps)
Skip connections at each level.
```

In [ ]:
class UNetGenerator(nn.Module):
    def __init__(self):
        super().__init__()

        # ---- ENCODER ----
        self.enc1 = nn.Sequential(  # 28 -> 14
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
        )
        self.enc2 = nn.Sequential(  # 14 -> 7
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
        )
        self.enc3 = nn.Sequential(  # 7 -> 3 (7/2 = 3 with padding)
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
        )
        self.enc4 = nn.Sequential(  # 3 -> 1
            nn.Conv2d(256, 512, 3, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2),
        )

        # ---- DECODER ----
        # Input channels doubled due to skip connections (except first)
        self.dec1 = nn.Sequential(  # 1 -> 3
            nn.ConvTranspose2d(512, 256, 3, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
        )
        self.dec2 = nn.Sequential(  # 3 -> 7 (but we need to handle odd sizes)
            nn.ConvTranspose2d(256 + 256, 128, 4, 2, 1, output_padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
        )
        self.dec3 = nn.Sequential(  # 7 -> 14
            nn.ConvTranspose2d(128 + 128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        self.dec4 = nn.Sequential(  # 14 -> 28
            nn.ConvTranspose2d(64 + 64, 1, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        # Encode (save for skip connections)
        e1 = self.enc1(x)    # (b, 64, 14, 14)
        e2 = self.enc2(e1)   # (b, 128, 7, 7)
        e3 = self.enc3(e2)   # (b, 256, 3, 3)
        e4 = self.enc4(e3)   # (b, 512, 1, 1)  <- bottleneck

        # Decode with skip connections
        d1 = self.dec1(e4)                   # (b, 256, 3, 3)
        d1 = torch.cat([d1, e3], dim=1)      # (b, 512, 3, 3)  <- skip!

        d2 = self.dec2(d1)                   # (b, 128, 7, 7)
        d2 = torch.cat([d2, e2], dim=1)      # (b, 256, 7, 7)  <- skip!

        d3 = self.dec3(d2)                   # (b, 64, 14, 14)
        d3 = torch.cat([d3, e1], dim=1)      # (b, 128, 14, 14) <- skip!

        out = self.dec4(d3)                  # (b, 1, 28, 28)
        return out


# Test
gen = UNetGenerator().to(device)
test_input = torch.randn(2, 1, 28, 28, device=device)
test_output = gen(test_input)
print(f"U-Net: {test_input.shape} -> {test_output.shape}")
print(f"Parameters: {sum(p.numel() for p in gen.parameters()):,}")

## 3. PatchGAN Discriminator

Outputs a grid of scores. Sees input + output concatenated (2 channels for MNIST).

In [ ]:
class PatchGAN(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: 2 channels (edge map + digit, concatenated)
        self.model = nn.Sequential(
            nn.Conv2d(2, 64, 4, 2, 1),           # 28 -> 14
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1, bias=False),  # 14 -> 7
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 1, 4, 1, 1),           # 7 -> 6
            # No sigmoid — use BCEWithLogitsLoss
        )

    def forward(self, input_img, output_img):
        # Concatenate input and output along channel dimension
        x = torch.cat([input_img, output_img], dim=1)  # (b, 2, 28, 28)
        return self.model(x)  # (b, 1, 6, 6) — grid of patch scores


# Test
disc = PatchGAN().to(device)
edge = torch.randn(2, 1, 28, 28, device=device)
digit = torch.randn(2, 1, 28, 28, device=device)
out = disc(edge, digit)
print(f"PatchGAN: edge{edge.shape} + digit{digit.shape} -> scores{out.shape}")
print(f"Each cell = one local patch's real/fake score")
print(f"Parameters: {sum(p.numel() for p in disc.parameters()):,}")

## 4. Training Loop

The Pix2Pix recipe:
- **G loss** = GAN loss (fool D) + 100 * L1 loss (match target)
- **D loss** = real pairs vs fake pairs
- Adam with lr=2e-4, betas=(0.5, 0.999)

**Predict first:** the G loss starts large. Which term dominates it — the GAN term or 100 × L1? (Expected: 100 × L1.)

In [ ]:
# Fresh models
gen  = UNetGenerator().to(device)
disc = PatchGAN().to(device)

opt_G = torch.optim.Adam(gen.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(disc.parameters(), lr=2e-4, betas=(0.5, 0.999))

bce = nn.BCEWithLogitsLoss()
l1  = nn.L1Loss()
lambda_L1 = 100

epochs = 20
d_losses, g_losses = [], []

print(f"Training Pix2Pix for {epochs} epochs...")
print(f"Lambda L1 = {lambda_L1}")
print()

In [ ]:
for epoch in range(epochs):
    d_loss_sum, g_loss_sum = 0, 0
    n_batches = 0

    for edges, targets, _ in train_loader:
        edges   = edges.to(device)
        targets = targets.to(device)

        # --- Discriminator step ---
        fake = gen(edges).detach()

        # Real pair: (edges, real target)
        d_real = disc(edges, targets)
        loss_D_real = bce(d_real, torch.ones_like(d_real))

        # Fake pair: (edges, generated output)
        d_fake = disc(edges, fake)
        loss_D_fake = bce(d_fake, torch.zeros_like(d_fake))

        loss_D = (loss_D_real + loss_D_fake) / 2

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # --- Generator step ---
        fake = gen(edges)

        d_fake = disc(edges, fake)
        loss_GAN = bce(d_fake, torch.ones_like(d_fake))  # fool D
        loss_L1  = l1(fake, targets)                      # match target

        loss_G = loss_GAN + lambda_L1 * loss_L1

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        d_loss_sum += loss_D.item()
        g_loss_sum += loss_G.item()
        n_batches += 1

    avg_d = d_loss_sum / n_batches
    avg_g = g_loss_sum / n_batches
    d_losses.append(avg_d)
    g_losses.append(avg_g)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | D: {avg_d:.4f} | G: {avg_g:.4f} | L1: {loss_L1:.4f}")

print("\nDone!")

In [ ]:
# Loss curves
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(d_losses, label="D loss", alpha=0.8)
ax.plot(g_losses, label="G loss", alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Pix2Pix Training")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Results: Edge → Digit Translation

Let's see how well our Pix2Pix translates edges into digits.

In [ ]:
# Test on unseen data
gen.eval()   # note: eval() also switches dropout OFF; the paper keeps it ON at test time — try gen.train() here and compare
test_edges, test_targets, test_labels = next(iter(test_loader))
test_edges = test_edges.to(device)

with torch.no_grad():
    test_fake = gen(test_edges).cpu()

test_edges_cpu = test_edges.cpu()
n_show = 8

fig, axes = plt.subplots(3, n_show, figsize=(12, 5))
for i in range(n_show):
    axes[0, i].imshow((test_edges_cpu[i].squeeze() + 1)/2, cmap="gray")
    axes[0, i].axis("off")
    axes[0, i].set_title(str(test_labels[i].item()))

    axes[1, i].imshow((test_fake[i].squeeze() + 1)/2, cmap="gray")
    axes[1, i].axis("off")

    axes[2, i].imshow((test_targets[i].squeeze() + 1)/2, cmap="gray")
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Input\n(edges)", fontsize=11, rotation=0, labelpad=40)
axes[1, 0].set_ylabel("Pix2Pix\noutput", fontsize=11, rotation=0, labelpad=40)
axes[2, 0].set_ylabel("Ground\ntruth", fontsize=11, rotation=0, labelpad=40)

plt.suptitle("Pix2Pix: Edges → Digits (Test Set)", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Experiment 1: L1 Only vs L1 + GAN

The key claim from L16: L1 alone → blurry. L1 + GAN → sharp.
Let's prove it.

**Predict first:** which model will have the LOWER L1 error on the test set? (Expected: often the L1-only model — yet it looks softer. Lower pixel error ≠ sharper image.)

In [ ]:
# Train a Generator with L1 ONLY (no GAN loss, no Discriminator)
gen_l1 = UNetGenerator().to(device)
opt_l1 = torch.optim.Adam(gen_l1.parameters(), lr=2e-4, betas=(0.5, 0.999))

print("Training L1-only model (no GAN loss)...")
for epoch in range(epochs):
    for edges, targets, _ in train_loader:
        edges, targets = edges.to(device), targets.to(device)

        fake = gen_l1(edges)
        loss = l1(fake, targets)  # ONLY L1 loss, no GAN

        opt_l1.zero_grad()
        loss.backward()
        opt_l1.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{epochs} | L1: {loss:.4f}")

print("Done!")

In [ ]:
# Compare L1-only vs Pix2Pix (L1 + GAN)
gen.eval()
gen_l1.eval()

with torch.no_grad():
    fake_pix2pix = gen(test_edges).cpu()
    fake_l1only  = gen_l1(test_edges).cpu()

n_show = 8
fig, axes = plt.subplots(4, n_show, figsize=(12, 6))

for i in range(n_show):
    axes[0, i].imshow((test_edges_cpu[i].squeeze() + 1)/2, cmap="gray")
    axes[0, i].axis("off")

    axes[1, i].imshow((fake_l1only[i].squeeze() + 1)/2, cmap="gray")
    axes[1, i].axis("off")

    axes[2, i].imshow((fake_pix2pix[i].squeeze() + 1)/2, cmap="gray")
    axes[2, i].axis("off")

    axes[3, i].imshow((test_targets[i].squeeze() + 1)/2, cmap="gray")
    axes[3, i].axis("off")

axes[0, 0].set_ylabel("Input\n(edges)", fontsize=10, rotation=0, labelpad=45)
axes[1, 0].set_ylabel("L1 only\n(blurry?)", fontsize=10, rotation=0, labelpad=45)
axes[2, 0].set_ylabel("L1+GAN\n(sharp?)", fontsize=10, rotation=0, labelpad=45)
axes[3, 0].set_ylabel("Ground\ntruth", fontsize=10, rotation=0, labelpad=45)

plt.suptitle("L1 Only vs Pix2Pix (L1 + GAN)", fontsize=14)
plt.tight_layout()
plt.show()

# Compute L1 errors
l1_error_pix2pix = F.l1_loss(fake_pix2pix, test_targets).item()
l1_error_l1only  = F.l1_loss(fake_l1only, test_targets).item()
print(f"L1 error (L1 only):  {l1_error_l1only:.4f}")
print(f"L1 error (Pix2Pix):  {l1_error_pix2pix:.4f}")
print("\nL1-only may have lower L1 error but look blurrier.")
print("Pix2Pix may have slightly higher L1 but look sharper.")

## 7. Experiment 2: What the PatchGAN Sees

Let's visualize the Discriminator's patch-level scores. Each cell of the 6×6 map sees a 22×22 patch of the 28×28 image.

In [ ]:
disc.eval()

with torch.no_grad():
    # Real pair scores
    d_real_scores = disc(test_edges, test_targets.to(device)).cpu()
    # Fake pair scores
    d_fake_scores = disc(test_edges, gen(test_edges)).cpu()

n_show = 4
fig, axes = plt.subplots(4, n_show, figsize=(10, 10))

for i in range(n_show):
    # Row 0: Real target
    axes[0, i].imshow((test_targets[i].squeeze() + 1)/2, cmap="gray")
    axes[0, i].axis("off")

    # Row 1: D's scores for real pair
    scores_real = torch.sigmoid(d_real_scores[i].squeeze())
    im = axes[1, i].imshow(scores_real, cmap="RdYlGn", vmin=0, vmax=1)
    axes[1, i].axis("off")
    axes[1, i].set_title(f"avg={scores_real.mean():.2f}", fontsize=9)

    # Row 2: Fake output
    axes[2, i].imshow((fake_pix2pix[i].squeeze() + 1)/2, cmap="gray")
    axes[2, i].axis("off")

    # Row 3: D's scores for fake pair
    scores_fake = torch.sigmoid(d_fake_scores[i].squeeze())
    axes[3, i].imshow(scores_fake, cmap="RdYlGn", vmin=0, vmax=1)
    axes[3, i].axis("off")
    axes[3, i].set_title(f"avg={scores_fake.mean():.2f}", fontsize=9)

axes[0, 0].set_ylabel("Real\ntarget", fontsize=10, rotation=0, labelpad=40)
axes[1, 0].set_ylabel("D scores\n(real)", fontsize=10, rotation=0, labelpad=40)
axes[2, 0].set_ylabel("Fake\noutput", fontsize=10, rotation=0, labelpad=40)
axes[3, 0].set_ylabel("D scores\n(fake)", fontsize=10, rotation=0, labelpad=40)

plt.suptitle("PatchGAN Discriminator Scores (green=real, red=fake)", fontsize=13)
plt.tight_layout()
plt.show()

print("Each cell in the score map = one patch's real/fake judgment.")
print("Green patches = D thinks real. Red patches = D thinks fake.")

## 8. Experiment 3: Skip Connections Matter

What if we remove skip connections? Just encoder-decoder without the U.

**Predict first:** which strokes will the no-skip model lose first — thin ends or the overall shape?

In [ ]:
class NoSkipGenerator(nn.Module):
    """Same architecture as UNet but WITHOUT skip connections."""
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2))
        self.enc2 = nn.Sequential(nn.Conv2d(64, 128, 4, 2, 1, bias=False),
                                  nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.enc3 = nn.Sequential(nn.Conv2d(128, 256, 4, 2, 1, bias=False),
                                  nn.BatchNorm2d(256), nn.LeakyReLU(0.2))
        self.enc4 = nn.Sequential(nn.Conv2d(256, 512, 3, 1, 0, bias=False),
                                  nn.BatchNorm2d(512), nn.LeakyReLU(0.2))

        # NO skip connections → no doubled channels
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(512, 256, 3, 1, 0, bias=False),
                                  nn.BatchNorm2d(256), nn.ReLU(), nn.Dropout(0.5))
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(256, 128, 4, 2, 1,
                                  output_padding=1, bias=False),
                                  nn.BatchNorm2d(128), nn.ReLU(), nn.Dropout(0.5))
        self.dec3 = nn.Sequential(nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
                                  nn.BatchNorm2d(64), nn.ReLU())
        self.dec4 = nn.Sequential(nn.ConvTranspose2d(64, 1, 4, 2, 1), nn.Tanh())

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        # Decode WITHOUT skip connections
        d1 = self.dec1(e4)
        d2 = self.dec2(d1)     # no cat with e2
        d3 = self.dec3(d2)     # no cat with e1
        return self.dec4(d3)


# Train it
gen_noskip = NoSkipGenerator().to(device)
disc_noskip = PatchGAN().to(device)
opt_G2 = torch.optim.Adam(gen_noskip.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D2 = torch.optim.Adam(disc_noskip.parameters(), lr=2e-4, betas=(0.5, 0.999))

print("Training encoder-decoder WITHOUT skip connections...")
for epoch in range(epochs):
    for edges, targets, _ in train_loader:
        edges, targets = edges.to(device), targets.to(device)

        # D step
        fake = gen_noskip(edges).detach()
        d_real, d_fake = disc_noskip(edges, targets), disc_noskip(edges, fake)
        loss_D = (bce(d_real, torch.ones_like(d_real)) + bce(d_fake, torch.zeros_like(d_fake))) / 2
        opt_D2.zero_grad(); loss_D.backward(); opt_D2.step()

        # G step
        fake = gen_noskip(edges)
        d_fake = disc_noskip(edges, fake)
        loss_G = bce(d_fake, torch.ones_like(d_fake)) + lambda_L1 * l1(fake, targets)
        opt_G2.zero_grad(); loss_G.backward(); opt_G2.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{epochs} | D: {loss_D:.4f} | G: {loss_G:.4f}")

print("Done!")

In [ ]:
# Compare: U-Net vs No-Skip
gen.eval()
gen_noskip.eval()

with torch.no_grad():
    fake_unet   = gen(test_edges).cpu()
    fake_noskip = gen_noskip(test_edges).cpu()

n_show = 8
fig, axes = plt.subplots(4, n_show, figsize=(12, 6))

for i in range(n_show):
    axes[0, i].imshow((test_edges_cpu[i].squeeze() + 1)/2, cmap="gray")
    axes[0, i].axis("off")

    axes[1, i].imshow((fake_noskip[i].squeeze() + 1)/2, cmap="gray")
    axes[1, i].axis("off")

    axes[2, i].imshow((fake_unet[i].squeeze() + 1)/2, cmap="gray")
    axes[2, i].axis("off")

    axes[3, i].imshow((test_targets[i].squeeze() + 1)/2, cmap="gray")
    axes[3, i].axis("off")

axes[0, 0].set_ylabel("Input\n(edges)", fontsize=10, rotation=0, labelpad=45)
axes[1, 0].set_ylabel("No skip\n(blurry?)", fontsize=10, rotation=0, labelpad=45)
axes[2, 0].set_ylabel("U-Net\n(sharp!)", fontsize=10, rotation=0, labelpad=45)
axes[3, 0].set_ylabel("Ground\ntruth", fontsize=10, rotation=0, labelpad=45)

plt.suptitle("Encoder-Decoder (no skip) vs U-Net (with skip)", fontsize=14)
plt.tight_layout()
plt.show()

l1_noskip = F.l1_loss(fake_noskip, test_targets).item()
l1_unet = F.l1_loss(fake_unet, test_targets).item()
print(f"L1 error (no skip): {l1_noskip:.4f}")
print(f"L1 error (U-Net):   {l1_unet:.4f}")
print("\nSkip connections preserve fine spatial details!")

## 9. Experiment 4: What the Skip Connections Carry

Let's visualize the encoder feature maps that get passed through skip connections.

In [ ]:
# Hook into the U-Net to grab encoder feature maps
gen.eval()
sample_edge = test_edges[:1]  # One edge map

with torch.no_grad():
    e1 = gen.enc1(sample_edge)
    e2 = gen.enc2(e1)
    e3 = gen.enc3(e2)
    e4 = gen.enc4(e3)  # bottleneck

# Visualize encoder feature maps at each level
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

# Original edge input
axes[0, 0].imshow((sample_edge.cpu().squeeze() + 1)/2, cmap="gray")
axes[0, 0].set_title("Input (28x28)", fontsize=10)
axes[0, 0].axis("off")

# Encoder layer 1
feat = e1.cpu().squeeze()  # (64, 14, 14)
axes[0, 1].imshow(feat.mean(dim=0), cmap="viridis")
axes[0, 1].set_title(f"enc1 (14x14, 64ch)", fontsize=10)
axes[0, 1].axis("off")

# Encoder layer 2
feat = e2.cpu().squeeze()
axes[0, 2].imshow(feat.mean(dim=0), cmap="viridis")
axes[0, 2].set_title(f"enc2 (7x7, 128ch)", fontsize=10)
axes[0, 2].axis("off")

# Encoder layer 3
feat = e3.cpu().squeeze()
axes[0, 3].imshow(feat.mean(dim=0), cmap="viridis")
axes[0, 3].set_title(f"enc3 (3x3, 256ch)", fontsize=10)
axes[0, 3].axis("off")

# Show individual channels of enc1 (the skip connection)
enc1_feats = e1.cpu().squeeze()  # (64, 14, 14)
for i in range(4):
    axes[1, i].imshow(enc1_feats[i], cmap="viridis")
    axes[1, i].set_title(f"enc1 ch.{i}", fontsize=10)
    axes[1, i].axis("off")

plt.suptitle("What Skip Connections Carry: Encoder Feature Maps", fontsize=13)
plt.tight_layout()
plt.show()

print("Top: mean activation at each encoder level")
print("Bottom: individual channels of enc1 (passed via skip to decoder)")
print("\nNotice: early layers capture EDGES and POSITIONS.")
print("These spatial details would be lost in the 1x1 bottleneck.")

## Key Takeaways

1. **Paired data creation** — we made edge↔digit pairs from MNIST
2. **U-Net skip connections** — concatenate encoder features with decoder features
3. **PatchGAN** — outputs a grid of local scores, not one global score
4. **Combined loss** — GAN + 100*L1 = sharp AND faithful
5. **L1 only → blurry** — confirmed experimentally!
6. **No skip → loses detail** — confirmed experimentally!
7. **Conditional D** — sees input+output together, judges if they match

**Your turn:** set `lambda_L1` to 1 and to 1000 and retrain. Which run hallucinates strokes, and which one blurs?

**Next: L18 — Data Augmentation & Privacy with GANs**